# 03 - Résultats du Traitement des Données Texte

## Section : Gestion du Texte

### Objectif
Valider les résultats du preprocessing effectué dans le notebook 02 : statistiques avant/après, visualisations, contrôles de qualité.

**Prérequis** : Exécuter d'abord `02_texte_traitement_donnees.ipynb` pour générer les fichiers dans `data/processed/`.




In [1]:
# Chargement des données (nécessite d'avoir exécuté 02_texte_traitement_donnees)
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.append(str(Path('../').resolve()))

from src.preprocessing import has_html
from src.utils.data_loader import load_data

DATA_DIR = Path('../data brut')
OUTPUT_DIR = Path('../data/processed')

# Données brutes (pour comparaison avant)
X_train_original, y_train, X_test_original = load_data(data_dir=DATA_DIR)

# Données traitées (sortie du notebook 02)
X_train_final = pd.read_csv(OUTPUT_DIR / 'X_train_clean.csv')
X_test_final = pd.read_csv(OUTPUT_DIR / 'X_test_clean.csv')

print("✅ Données chargées pour validation")



🔄 Chargement des données...
✅ Données chargées avec succès !
  - X_train : 84,916 lignes × 5 colonnes
  - y_train : 84,916 lignes × 2 colonnes
  - X_test  : 13,812 lignes × 5 colonnes
✅ Données chargées pour validation


## 1. Statistiques et Validation

### 1.1 Comparaison avant/après preprocessing


In [2]:
# Données chargées dans la cellule précédente

print("="*80)
print("COMPARAISON AVANT / APRÈS PREPROCESSING")
print("="*80)

# Statistiques avant
print("\n📊 AVANT PREPROCESSING :")
print(f"  - Descriptions avec HTML : {X_train_original['description'].apply(has_html).sum()} ({X_train_original['description'].apply(has_html).sum()/len(X_train_original)*100:.2f}%)")
print(f"  - Descriptions manquantes : {X_train_original['description'].isna().sum()} ({X_train_original['description'].isna().sum()/len(X_train_original)*100:.2f}%)")
print(f"  - Longueur moyenne description : {X_train_original['description'].fillna('').astype(str).str.len().mean():.1f} caractères")

# Statistiques après
# Vérifier le HTML sur les descriptions nettoyées (pas sur has_html qui est calculé avant nettoyage)
html_after_clean = X_train_final['description_clean'].apply(has_html).sum()
print("\n📊 APRÈS PREPROCESSING :")
print(f"  - Descriptions avec HTML restant : {html_after_clean} ({html_after_clean/len(X_train_final)*100:.2f}%)")
print(f"  - Descriptions vides : {(X_train_final['description_clean'] == '').sum()} ({(X_train_final['description_clean'] == '').sum()/len(X_train_final)*100:.2f}%)")
print(f"  - Longueur moyenne text_combined : {X_train_final['text_combined'].str.len().mean():.1f} caractères")
print(f"  - Features créées : {len([c for c in X_train_final.columns if c not in ['productid', 'imageid', 'designation_clean', 'description_clean', 'text_combined']])}")



COMPARAISON AVANT / APRÈS PREPROCESSING

📊 AVANT PREPROCESSING :
  - Descriptions avec HTML : 15645 (18.42%)
  - Descriptions manquantes : 29800 (35.09%)
  - Longueur moyenne description : 524.6 caractères

📊 APRÈS PREPROCESSING :
  - Descriptions avec HTML restant : 29 (0.03%)
  - Descriptions vides : 0 (0.00%)
  - Longueur moyenne text_combined : 559.2 caractères
  - Features créées : 12


### 1.2 Visualisations de validation


In [3]:
# Visualisation 1 : Distribution des longueurs avant/après
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Avant
desc_lengths_before = X_train_original['description'].fillna('').astype(str).str.len()
axes[0].hist(desc_lengths_before[desc_lengths_before < 2000], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Longueur (caractères)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution des longueurs de descriptions (AVANT preprocessing)')
axes[0].axvline(desc_lengths_before.median(), color='r', linestyle='--', label=f'Médiane: {desc_lengths_before.median():.0f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Après
desc_lengths_after = X_train_final['text_combined'].str.len()
axes[1].hist(desc_lengths_after[desc_lengths_after < 2000], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Longueur (caractères)')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution des longueurs de textes combinés (APRÈS preprocessing)')
axes[1].axvline(desc_lengths_after.median(), color='r', linestyle='--', label=f'Médiane: {desc_lengths_after.median():.0f}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()



NameError: name 'plt' is not defined

In [ ]:
# Visualisation 2 : Features créées
feature_cols_numeric = [
    'designation_length', 'description_length', 'total_text_length',
    'designation_word_count', 'description_word_count', 'total_word_count',
    'text_completeness', 'description_quality_score', 'word_density'
]

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, col in enumerate(feature_cols_numeric):
    if col in X_train_final.columns:
        axes[idx].hist(X_train_final[col], bins=50, edgecolor='black', alpha=0.7)
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Fréquence')
        axes[idx].set_title(f'Distribution de {col}')
        axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()



### 7.3 Validation finale

Vérification de la cohérence des données et absence d'erreurs.


In [ ]:
# Validation finale
print("="*80)
print("VALIDATION FINALE")
print("="*80)

# Vérifications
checks = []

# 1. Pas de NaN dans les colonnes importantes
checks.append(('Pas de NaN dans text_combined', X_train_final['text_combined'].isna().sum() == 0))
checks.append(('Pas de NaN dans designation_clean', X_train_final['designation_clean'].isna().sum() == 0))

# 2. Pas de textes vides
checks.append(('Pas de textes vides', (X_train_final['text_combined'] == '').sum() == 0))

# 3. Cohérence train/test
checks.append(('Même nombre de colonnes train/test', X_train_final.shape[1] == X_test_final.shape[1]))
checks.append(('Colonnes identiques train/test', list(X_train_final.columns) == list(X_test_final.columns)))

# 4. Pas de HTML restant (vérifier sur les descriptions nettoyées)
html_remaining_clean = X_train_final['description_clean'].apply(has_html).sum()
checks.append(('Pas de HTML restant dans description_clean', html_remaining_clean == 0))

# Afficher les résultats
print("\n✅ Résultats des vérifications :")
for check_name, check_result in checks:
    status = "✅" if check_result else "❌"
    print(f"  {status} {check_name}")

all_passed = all(result for _, result in checks)
if all_passed:
    print("\n🎉 Toutes les validations sont passées ! Le dataset est prêt pour la modélisation.")
else:
    print("\n⚠️  Certaines validations ont échoué. Vérifiez les données.")
    
# Si HTML restant, investiguer
if html_remaining_clean > 0:
    print(f"\n🔍 Investigation des {html_remaining_clean} descriptions avec HTML restant...")



### 7.4 Investigation du HTML Restant

Analyse des descriptions qui contiennent encore du HTML après nettoyage.


In [ ]:
# Identifier les descriptions avec HTML restant
html_remaining_mask = X_train_final['description_clean'].apply(has_html)
html_remaining_descriptions = X_train_final[html_remaining_mask].copy()

print(f"📊 Analyse des {len(html_remaining_descriptions)} descriptions avec HTML restant :\n")

# S'assurer que X_train est accessible (recharger si nécessaire)
if 'X_train' not in locals() or X_train is None:
    X_train, _, _ = load_data(data_dir=DATA_DIR)

# Afficher quelques exemples
print("="*80)
print("EXEMPLES DE DESCRIPTIONS AVEC HTML RESTANT")
print("="*80)

for idx, row in html_remaining_descriptions.head(10).iterrows():
    print(f"\n--- Exemple {idx} ---")
    print(f"ProductID: {row['productid']}")
    print(f"Description originale (premiers 300 caractères):")
    # Récupérer la description originale depuis X_train
    original_desc = X_train.loc[X_train['productid'] == row['productid'], 'description'].values
    if len(original_desc) > 0:
        print(f"  {str(original_desc[0])[:300]}...")
    print(f"\nDescription nettoyée (premiers 300 caractères):")
    print(f"  {row['description_clean'][:300]}...")
    print(f"\nHTML détecté dans description_clean: {has_html(row['description_clean'])}")
    
    # Extraire les tags HTML restants
    import re
    html_tags = re.findall(r'<[^>]+>', row['description_clean'])
    if html_tags:
        print(f"Tags HTML trouvés: {html_tags[:5]}")  # Limiter à 5 tags


In [ ]:
# Analyser les patterns de HTML restant
import re
from collections import Counter

print("\n" + "="*80)
print("ANALYSE DES PATTERNS DE HTML RESTANT")
print("="*80)

# Extraire tous les tags HTML restants
all_remaining_tags = []
all_remaining_texts = []

for desc in html_remaining_descriptions['description_clean']:
    if has_html(desc):
        tags = re.findall(r'<[^>]+>', desc)
        all_remaining_tags.extend(tags)
        all_remaining_texts.append(desc)

# Compter les tags les plus fréquents
tag_counter = Counter(all_remaining_tags)
print(f"\n📊 Top 20 tags HTML les plus fréquents dans les descriptions restantes :")
for tag, count in tag_counter.most_common(20):
    print(f"  {tag}: {count} occurrences")

# Analyser les types de tags
print(f"\n📊 Types de tags HTML restants :")
tag_types = {}
for tag in all_remaining_tags:
    tag_name = re.match(r'</?(\w+)', tag)
    if tag_name:
        tag_type = tag_name.group(1)
        tag_types[tag_type] = tag_types.get(tag_type, 0) + 1

for tag_type, count in sorted(tag_types.items(), key=lambda x: x[1], reverse=True):
    print(f"  {tag_type}: {count} occurrences")


In [ ]:
# Tester pourquoi BeautifulSoup n'a pas nettoyé ces cas
from bs4 import BeautifulSoup

print("\n" + "="*80)
print("TEST DE NETTOYAGE AVEC BEAUTIFULSOUP")
print("="*80)

# S'assurer que X_train est accessible
if 'X_train' not in locals() or X_train is None:
    X_train, _, _ = load_data(data_dir=DATA_DIR)

# Prendre quelques exemples et tester le nettoyage
test_examples = html_remaining_descriptions.head(5)

for idx, row in test_examples.iterrows():
    print(f"\n--- Test {idx} ---")
    original_desc = X_train.loc[X_train['productid'] == row['productid'], 'description'].values
    if len(original_desc) > 0:
        original_desc = str(original_desc[0])
    else:
        original_desc = "Non trouvé"
    cleaned_desc = row['description_clean']
    
    print(f"Description originale (premiers 200 caractères):")
    print(f"  {original_desc[:200]}...")
    
    print(f"\nDescription nettoyée actuelle (premiers 200 caractères):")
    print(f"  {cleaned_desc[:200]}...")
    
    # Tester un nettoyage manuel
    try:
        soup = BeautifulSoup(cleaned_desc, 'html.parser')
        manual_clean = soup.get_text(separator=' ', strip=True)
        print(f"\nNettoyage manuel avec BeautifulSoup (premiers 200 caractères):")
        print(f"  {manual_clean[:200]}...")
        
        # Vérifier si le nettoyage manuel a fonctionné
        if has_html(manual_clean):
            print(f"  ⚠️  HTML toujours présent après nettoyage manuel")
            # Extraire les tags restants
            remaining_tags = re.findall(r'<[^>]+>', manual_clean)
            print(f"  Tags restants: {remaining_tags[:5]}")
        else:
            print(f"  ✅ HTML supprimé avec succès")
    except Exception as e:
        print(f"  ❌ Erreur lors du nettoyage: {e}")


In [ ]:
# Analyser les cas où le HTML est dans des attributs ou malformé
print("\n" + "="*80)
print("ANALYSE DES CAS PARTICULIERS")
print("="*80)

# Chercher des patterns spécifiques
patterns_to_check = [
    (r'&lt;[^&]+&gt;', 'HTML encodé (&lt; et &gt;)'),
    (r'<[^>]*$', 'Balise HTML non fermée'),
    (r'[^<]*>', 'Caractère > isolé'),
    (r'<[^>]*[^>]', 'Balise HTML malformée'),
]

for pattern, description in patterns_to_check:
    matches = []
    for desc in html_remaining_descriptions['description_clean']:
        if re.search(pattern, desc):
            matches.append(desc)
    if matches:
        print(f"\n📌 {description}: {len(matches)} occurrences")
        if len(matches) <= 3:
            for match in matches:
                print(f"  Exemple: {match[:150]}...")

# Vérifier si c'est juste des caractères < ou > isolés
isolated_chars = 0
for desc in html_remaining_descriptions['description_clean']:
    # Compter les < et > qui ne sont pas dans des balises HTML valides
    isolated = re.findall(r'[^<]&lt;|&gt;[^>]', desc)
    if isolated:
        isolated_chars += 1

if isolated_chars > 0:
    print(f"\n📌 Caractères < ou > isolés ou encodés: {isolated_chars} descriptions")


In [ ]:
# Recommandations finales
print("\n" + "="*80)
print("RECOMMANDATIONS")
print("="*80)

total_with_html = len(html_remaining_descriptions)
percentage = (total_with_html / len(X_train_final)) * 100

print(f"\n📊 Résumé :")
print(f"  - Descriptions avec HTML restant : {total_with_html} ({percentage:.2f}%)")
print(f"  - Impact sur le dataset : Négligeable (< 0.2%)")

print(f"\n💡 Recommandations :")
print(f"  1. Le HTML restant représente seulement {percentage:.2f}% du dataset")
print(f"  2. Impact négligeable sur la modélisation")
print(f"  3. Options possibles :")
print(f"     a) Accepter ce niveau de nettoyage (recommandé)")
print(f"     b) Appliquer un nettoyage supplémentaire avec regex plus agressif")
print(f"     c) Analyser manuellement les cas restants si nécessaire")

print(f"\n✅ Conclusion : Le preprocessing est suffisant pour la modélisation.")
print(f"   Les {total_with_html} descriptions avec HTML restant ne devraient pas")
print(f"   impacter significativement les performances du modèle.")


## 2. Synthèse et Prochaines Étapes

### Résumé du Preprocessing

Le preprocessing a été effectué avec succès :
- ✅ HTML nettoyé des descriptions
- ✅ Valeurs manquantes gérées
- ✅ Textes normalisés
- ✅ Features supplémentaires créées
- ✅ Datasets prêts pour la modélisation

### Prochaines étapes

1. **Modélisation** : Utiliser les datasets nettoyés pour entraîner des modèles
2. **Vectorisation** : Transformer les textes en vecteurs (TF-IDF, Word2Vec, BERT, etc.)
3. **Sélection de modèle** : Tester différents algorithmes (Naive Bayes, SVM, Random Forest, etc.)
4. **Évaluation** : Mesurer les performances avec des métriques adaptées au déséquilibre

### Fichiers générés

- `data/processed/X_train_clean.csv` : Dataset d'entraînement nettoyé
- `data/processed/X_test_clean.csv` : Dataset de test nettoyé
- `data/processed/y_train.csv` : Labels d'entraînement

